# Sequential Goal Forecasting With Geometry-Aware Sequences

This notebook adds a sequence-oriented extension on top of the existing checkpoint panel.

It implements two aligned sequence views:

- `minute_hybrid`: 15-minute ordered bins before each checkpoint
- `possession_sequence`: strict possession summaries from run and shot events, with unmatched pressure retained as a residual summary

Outputs are exported to `artifacts/sequential_goal_forecasting_geometry/`.

In [1]:
from __future__ import annotations

import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, balanced_accuracy_score, brier_score_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore', category=RuntimeWarning)

ROOT = next(path for path in [Path.cwd(), Path.cwd().parent] if (path / 'data').exists())
DATA_DIR = ROOT / 'data'
ARTIFACT_DIR = ROOT / 'artifacts' / 'sequential_goal_forecasting_geometry'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
LOOKBACK_MINUTES = 15
FORWARD_ANGLE_DEG = 30
LATERAL_ANGLE_DEG = 120
PRESSURE_RICH_THRESHOLD = 2
LATE_BURST_THRESHOLD = 2

PERIOD_OFFSETS = {
    'half_1': 0,
    'half_2': 45,
    'extra_time_1': 90,
    'extra_time_2': 105,
}
CHECKPOINT_TO_CONT_MIN = {
    'H1_15': 15,
    'H1_30': 30,
    'H1_45': 45,
    'H2_15': 60,
    'H2_30': 75,
    'H2_45': 90,
    'ET1_15': 105,
}
CATEGORICAL_BASE_FEATURES = ['checkpoint', 'position', 'formation']
BASE_FEATURE_COLUMNS = [
    'checkpoint',
    'checkpoint_min',
    'position',
    'is_home',
    'formation',
    'minute_in',
    'minutes_available_before_checkpoint',
    'last15_sprints',
    'last15_hsr',
    'last15_distance',
    'last15_mean_max_speed',
    'last15_peak_speed',
    'last15_shots',
    'last15_shots_on_target',
    'last15_shots_under_press',
    'last15_shots_top_third',
    'cumul_sprints',
    'cumul_hsr',
    'cumul_distance',
    'cumul_mean_max_speed',
    'cumul_peak_speed',
    'cumul_shots',
    'cumul_shots_on_target',
    'cumul_shots_under_press',
    'cumul_shots_top_third',
]

In [2]:
def add_event_time(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out['event_cont_min'] = out['period'].map(PERIOD_OFFSETS) + pd.to_numeric(out['minute'], errors='coerce')
    return out


def safe_rate(numerator: float, denominator: float) -> float:
    return float(numerator / denominator) if denominator else 0.0


def circular_mean_deg(values: np.ndarray) -> float:
    if values.size == 0:
        return 0.0
    radians = np.deg2rad(values)
    return math.degrees(math.atan2(np.sin(radians).mean(), np.cos(radians).mean()))


def circular_diff_deg(a_deg: float, b_deg: float) -> float:
    diff = (a_deg - b_deg + 180.0) % 360.0 - 180.0
    return abs(diff)


def longest_true_streak(mask: pd.Series) -> int:
    best = 0
    current = 0
    for value in mask.astype(bool).tolist():
        current = current + 1 if value else 0
        best = max(best, current)
    return int(best)


def linear_slope(values: np.ndarray) -> float:
    if values.size <= 1:
        return 0.0
    x = np.arange(values.size, dtype=float)
    x = x - x.mean()
    y = values.astype(float) - values.astype(float).mean()
    denominator = np.square(x).sum()
    if denominator == 0:
        return 0.0
    return float((x * y).sum() / denominator)


def build_preprocessor(feature_columns: list[str]) -> ColumnTransformer:
    categorical_columns = [column for column in feature_columns if column in CATEGORICAL_BASE_FEATURES]
    numeric_columns = [column for column in feature_columns if column not in categorical_columns]
    return ColumnTransformer(
        transformers=[
            (
                'num',
                Pipeline(
                    steps=[
                        ('imputer', SimpleImputer(strategy='constant', fill_value=0.0)),
                        ('scaler', StandardScaler()),
                    ]
                ),
                numeric_columns,
            ),
            (
                'cat',
                Pipeline(
                    steps=[
                        ('imputer', SimpleImputer(strategy='most_frequent')),
                        ('onehot', OneHotEncoder(handle_unknown='ignore')),
                    ]
                ),
                categorical_columns,
            ),
        ]
    )


def score_predictions(y_true: np.ndarray, pred_prob: np.ndarray) -> dict[str, float]:
    pred_label = (pred_prob >= 0.5).astype(int)
    metrics = {
        'balanced_accuracy': balanced_accuracy_score(y_true, pred_label),
        'pr_auc': average_precision_score(y_true, pred_prob),
        'brier_score': brier_score_loss(y_true, pred_prob),
    }
    metrics['roc_auc'] = roc_auc_score(y_true, pred_prob) if len(np.unique(y_true)) > 1 else np.nan
    return metrics


def subgroup_metric_table(predictions: pd.DataFrame, group_column: str) -> pd.DataFrame:
    rows = []
    for group_value, subset in predictions.groupby(group_column, dropna=False):
        row = {
            'group_type': group_column,
            'group_value': group_value,
            'rows': int(len(subset)),
            'positive_rate': float(subset['scored_after'].mean()),
            'balanced_accuracy': balanced_accuracy_score(
                subset['scored_after'],
                (subset['pred_prob'] >= 0.5).astype(int),
            ),
            'pr_auc': average_precision_score(subset['scored_after'], subset['pred_prob']),
            'brier_score': brier_score_loss(subset['scored_after'], subset['pred_prob']),
        }
        row['roc_auc'] = roc_auc_score(subset['scored_after'], subset['pred_prob']) if subset['scored_after'].nunique() > 1 else np.nan
        rows.append(row)
    return pd.DataFrame(rows).sort_values(['group_type', 'rows'], ascending=[True, False]).reset_index(drop=True)


def evaluate_feature_set(
    development_df: pd.DataFrame,
    feature_columns: list[str],
    model_name: str,
    return_oof: bool = False,
) -> tuple[pd.DataFrame, pd.DataFrame | None]:
    if model_name == 'logreg':
        model = LogisticRegression(
            max_iter=4000,
            class_weight='balanced',
            solver='liblinear',
            C=0.25,
            random_state=RANDOM_STATE,
        )
    elif model_name == 'rf':
        model = RandomForestClassifier(
            n_estimators=200,
            min_samples_leaf=5,
            class_weight='balanced_subsample',
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
    else:
        raise ValueError(f'Unsupported model_name: {model_name}')

    folds = sorted(development_df['fold'].dropna().unique())
    fold_rows = []
    oof_rows = []
    for fold in folds:
        train_df = development_df[development_df['fold'] != fold].copy()
        valid_df = development_df[development_df['fold'] == fold].copy()
        pipeline = Pipeline(
            steps=[
                ('prep', build_preprocessor(feature_columns)),
                ('model', model),
            ]
        )
        pipeline.fit(train_df[feature_columns], train_df['scored_after'].astype(int))
        pred_prob = pipeline.predict_proba(valid_df[feature_columns])[:, 1]
        fold_metrics = score_predictions(valid_df['scored_after'].to_numpy(), pred_prob)
        fold_rows.append({'fold': fold, **fold_metrics})

        if return_oof:
            oof_part = valid_df[
                ['player_appearance_id', 'fixture_id', 'checkpoint', 'position', 'scored_after']
            ].copy()
            oof_part['fold'] = fold
            oof_part['pred_prob'] = pred_prob
            oof_rows.append(oof_part)

    fold_metrics_df = pd.DataFrame(fold_rows)
    oof_df = pd.concat(oof_rows, ignore_index=True) if return_oof else None
    return fold_metrics_df, oof_df

In [3]:
base = pd.read_csv(DATA_DIR / 'players_quarters_final.csv')
pressure = pd.read_csv(DATA_DIR / 'player_appearance_behaviour_under_pressure.csv')
runs = pd.read_csv(DATA_DIR / 'player_appearance_run.csv')
shots = pd.read_csv(DATA_DIR / 'player_appearance_shot_limited.csv')
fixture_assignments = pd.read_csv(DATA_DIR / 'splits' / 'fixture_assignments.csv')

base['checkpoint_cont_min'] = base['checkpoint'].map(CHECKPOINT_TO_CONT_MIN)
base['minutes_available_before_checkpoint'] = base['checkpoint_cont_min'] - base['minute_in'] + 1
base = base.merge(fixture_assignments, on='fixture_id', how='left', validate='many_to_one')

pressure = add_event_time(pressure)
pressure['pass_angle_observed'] = pressure['pass_angle'].notna().astype(int)
pressure['pass_angle_rad'] = np.deg2rad(pressure['pass_angle'])
pressure['pass_angle_sin'] = np.sin(pressure['pass_angle_rad']).fillna(0.0)
pressure['pass_angle_cos'] = np.cos(pressure['pass_angle_rad']).fillna(0.0)
pressure['pass_angle_abs'] = pressure['pass_angle'].abs()
pressure['forward_angle_flag'] = pressure['pass_angle_abs'].le(FORWARD_ANGLE_DEG).fillna(False).astype(int)
pressure['lateral_angle_flag'] = (
    pressure['pass_angle_abs'].gt(FORWARD_ANGLE_DEG).fillna(False)
    & pressure['pass_angle_abs'].le(LATERAL_ANGLE_DEG).fillna(False)
).astype(int)
pressure['backward_angle_flag'] = pressure['pass_angle_abs'].gt(LATERAL_ANGLE_DEG).fillna(False).astype(int)
pressure['turnover_flag'] = pressure['press_induced_outcome'].eq('turnover').astype(int)

runs = add_event_time(runs)

shots = add_event_time(shots)
shots = shots[shots['own_goal_player_appearance_id'].isna()].copy()
shots['under_pressure_flag'] = shots['under_pressure'].astype(int)
shots['top_stage_flag'] = shots['stage'].eq('top').astype(int)

strict_possession_map = (
    runs.groupby(['player_appearance_id', 'period', 'minute'], dropna=False)
    .agg(possession_nunique=('possession', 'nunique'), strict_possession=('possession', 'first'))
    .reset_index()
)
strict_possession_map.loc[
    strict_possession_map['possession_nunique'] != 1,
    'strict_possession',
] = np.nan

pressure = pressure.merge(
    strict_possession_map[
        ['player_appearance_id', 'period', 'minute', 'strict_possession', 'possession_nunique']
    ],
    on=['player_appearance_id', 'period', 'minute'],
    how='left',
)
pressure.rename(columns={'strict_possession': 'linked_possession'}, inplace=True)
pressure['linked_pressure_flag'] = pressure['linked_possession'].notna().astype(int)

display(Markdown('### Loaded Tables'))
display(
    pd.DataFrame(
        [
            {'table': 'players_quarters_final', 'rows': len(base), 'columns': base.shape[1]},
            {'table': 'player_appearance_behaviour_under_pressure', 'rows': len(pressure), 'columns': pressure.shape[1]},
            {'table': 'player_appearance_run', 'rows': len(runs), 'columns': runs.shape[1]},
            {'table': 'player_appearance_shot_limited', 'rows': len(shots), 'columns': shots.shape[1]},
        ]
    )
)

### Loaded Tables

,table,rows,columns
0,players_quarters_final,3486,37
1,player_appearance_behaviour_under_pressure,12185,23
2,player_appearance_run,35133,11
3,player_appearance_shot_limited,779,15


In [4]:
pressure_groups = {
    player_appearance_id: frame.sort_values(['event_cont_min', 'id']).reset_index(drop=True)
    for player_appearance_id, frame in pressure.groupby('player_appearance_id')
}
run_groups = {
    player_appearance_id: frame.sort_values(['event_cont_min', 'id']).reset_index(drop=True)
    for player_appearance_id, frame in runs.groupby('player_appearance_id')
}
shot_groups = {
    player_appearance_id: frame.sort_values(['event_cont_min', 'id']).reset_index(drop=True)
    for player_appearance_id, frame in shots.groupby('player_appearance_id')
}

sequence_rows = []
minute_sequence_rows = []
possession_sequence_rows = []

for row in base.itertuples(index=False):
    checkpoint_time = int(row.checkpoint_cont_min)
    history_start = max(1, checkpoint_time - LOOKBACK_MINUTES + 1)
    player_appearance_id = int(row.player_appearance_id)

    player_pressure = pressure_groups.get(player_appearance_id)
    player_runs = run_groups.get(player_appearance_id)
    player_shots = shot_groups.get(player_appearance_id)

    pressure_last15 = (
        player_pressure[
            (player_pressure['event_cont_min'] >= history_start)
            & (player_pressure['event_cont_min'] <= checkpoint_time)
        ].copy()
        if player_pressure is not None
        else pd.DataFrame(columns=pressure.columns)
    )
    runs_last15 = (
        player_runs[
            (player_runs['event_cont_min'] >= history_start)
            & (player_runs['event_cont_min'] <= checkpoint_time)
        ].copy()
        if player_runs is not None
        else pd.DataFrame(columns=runs.columns)
    )
    shots_last15 = (
        player_shots[
            (player_shots['event_cont_min'] >= history_start)
            & (player_shots['event_cont_min'] <= checkpoint_time)
        ].copy()
        if player_shots is not None
        else pd.DataFrame(columns=shots.columns)
    )

    pressure_pre = (
        player_pressure[player_pressure['event_cont_min'] <= checkpoint_time].copy()
        if player_pressure is not None
        else pd.DataFrame(columns=pressure.columns)
    )
    runs_pre = (
        player_runs[player_runs['event_cont_min'] <= checkpoint_time].copy()
        if player_runs is not None
        else pd.DataFrame(columns=runs.columns)
    )
    shots_pre = (
        player_shots[player_shots['event_cont_min'] <= checkpoint_time].copy()
        if player_shots is not None
        else pd.DataFrame(columns=shots.columns)
    )

    minute_rows = []
    for minute in range(history_start, checkpoint_time + 1):
        pressure_min = pressure_last15[pressure_last15['event_cont_min'] == minute]
        runs_min = runs_last15[runs_last15['event_cont_min'] == minute]
        shots_min = shots_last15[shots_last15['event_cont_min'] == minute]

        combined_possessions = pd.concat(
            [runs_min[['possession']], shots_min[['possession']]],
            ignore_index=True,
        )
        possession_transition_count = (
            max(int(combined_possessions['possession'].nunique()) - 1, 0)
            if not combined_possessions.empty
            else 0
        )

        minute_row = {
            'player_appearance_id': player_appearance_id,
            'checkpoint': row.checkpoint,
            'fixture_id': row.fixture_id,
            'checkpoint_cont_min': checkpoint_time,
            'minute_bin_cont_min': minute,
            'minute_bin_idx': minute - history_start + 1,
            'pressure_count': int(len(pressure_min)),
            'pressure_turnover_count': int(pressure_min['turnover_flag'].sum()) if not pressure_min.empty else 0,
            'pressure_angle_observed_count': int(pressure_min['pass_angle_observed'].sum()) if not pressure_min.empty else 0,
            'pressure_mean_sin': float(
                pressure_min.loc[pressure_min['pass_angle_observed'] == 1, 'pass_angle_sin'].mean()
            )
            if int(pressure_min['pass_angle_observed'].sum())
            else 0.0,
            'pressure_mean_cos': float(
                pressure_min.loc[pressure_min['pass_angle_observed'] == 1, 'pass_angle_cos'].mean()
            )
            if int(pressure_min['pass_angle_observed'].sum())
            else 0.0,
            'pressure_forward_share': float(
                pressure_min.loc[pressure_min['pass_angle_observed'] == 1, 'forward_angle_flag'].mean()
            )
            if int(pressure_min['pass_angle_observed'].sum())
            else 0.0,
            'pressure_lateral_share': float(
                pressure_min.loc[pressure_min['pass_angle_observed'] == 1, 'lateral_angle_flag'].mean()
            )
            if int(pressure_min['pass_angle_observed'].sum())
            else 0.0,
            'pressure_forward_count': int(pressure_min['forward_angle_flag'].sum()) if not pressure_min.empty else 0,
            'run_count': int(len(runs_min)),
            'run_distance_sum': float(runs_min['distance'].sum()) if not runs_min.empty else 0.0,
            'run_mean_speed': float(runs_min['max_speed'].mean()) if not runs_min.empty else 0.0,
            'run_max_speed': float(runs_min['max_speed'].max()) if not runs_min.empty else 0.0,
            'run_top_share': float(runs_min['stage'].eq('top').mean()) if not runs_min.empty else 0.0,
            'run_middle_share': float(runs_min['stage'].eq('middle').mean()) if not runs_min.empty else 0.0,
            'run_bottom_share': float(runs_min['stage'].eq('bottom').mean()) if not runs_min.empty else 0.0,
            'shot_count': int(len(shots_min)),
            'shot_under_pressure_count': int(shots_min['under_pressure_flag'].sum()) if not shots_min.empty else 0,
            'shot_top_stage_count': int(shots_min['top_stage_flag'].sum()) if not shots_min.empty else 0,
            'possession_transition_count': possession_transition_count,
        }
        minute_rows.append(minute_row)
        minute_sequence_rows.append(minute_row)

    minute_df = pd.DataFrame(minute_rows)
    activity_total = minute_df[['pressure_count', 'run_count', 'shot_count']].sum(axis=1)
    pressured_mask = minute_df['pressure_count'] > 0
    pressure_rich_mask = minute_df['pressure_count'] >= PRESSURE_RICH_THRESHOLD
    forward_pressure_mask = minute_df['pressure_forward_count'] > 0
    pressure_or_previous_mask = pressured_mask | pressured_mask.shift(fill_value=False)
    forward_pressure_or_previous_mask = forward_pressure_mask | forward_pressure_mask.shift(fill_value=False)
    observed_angles = pressure_last15.loc[pressure_last15['pass_angle'].notna(), 'pass_angle'].to_numpy(dtype=float)

    if observed_angles.size:
        radians = np.deg2rad(observed_angles)
        angle_concentration = float(
            np.sqrt(np.square(np.sin(radians).sum()) + np.square(np.cos(radians).sum())) / observed_angles.size
        )
        early_angles = pressure_last15.loc[
            (pressure_last15['event_cont_min'] < checkpoint_time - LOOKBACK_MINUTES / 2 + 1)
            & pressure_last15['pass_angle'].notna(),
            'pass_angle',
        ].to_numpy(dtype=float)
        late_angles = pressure_last15.loc[
            (pressure_last15['event_cont_min'] >= checkpoint_time - LOOKBACK_MINUTES / 2 + 1)
            & pressure_last15['pass_angle'].notna(),
            'pass_angle',
        ].to_numpy(dtype=float)
        directional_change_deg = (
            circular_diff_deg(circular_mean_deg(late_angles), circular_mean_deg(early_angles))
            if early_angles.size and late_angles.size
            else 0.0
        )
    else:
        angle_concentration = 0.0
        directional_change_deg = 0.0

    rich_following_run_distance = minute_df.loc[
        pressure_rich_mask.shift(fill_value=False),
        'run_distance_sum',
    ]
    forward_shot_counts = minute_df.loc[forward_pressure_or_previous_mask, 'shot_count']
    forward_up_shot_counts = minute_df.loc[
        forward_pressure_or_previous_mask,
        'shot_under_pressure_count',
    ]
    weights = np.linspace(1.0 / len(minute_df), 1.0, len(minute_df))

    linked_pressure_pre = pressure_pre[pressure_pre['linked_pressure_flag'] == 1].copy()
    unmatched_pressure_pre = pressure_pre[pressure_pre['linked_pressure_flag'] == 0].copy()
    total_pressure_pre = int(len(pressure_pre))

    possession_parts = []
    if not runs_pre.empty:
        runs_part = runs_pre[['event_cont_min', 'possession', 'distance']].copy()
        runs_part['run_count'] = 1
        runs_part['shot_count'] = 0
        runs_part['linked_pressure_count'] = 0
        runs_part['forward_pressure_count'] = 0
        possession_parts.append(runs_part)
    if not shots_pre.empty:
        shots_part = shots_pre[['event_cont_min', 'possession']].copy()
        shots_part['distance'] = 0.0
        shots_part['run_count'] = 0
        shots_part['shot_count'] = 1
        shots_part['linked_pressure_count'] = 0
        shots_part['forward_pressure_count'] = 0
        possession_parts.append(shots_part)
    if not linked_pressure_pre.empty:
        pressure_part = linked_pressure_pre[['event_cont_min', 'linked_possession', 'forward_angle_flag']].copy()
        pressure_part.rename(columns={'linked_possession': 'possession'}, inplace=True)
        pressure_part['distance'] = 0.0
        pressure_part['run_count'] = 0
        pressure_part['shot_count'] = 0
        pressure_part['linked_pressure_count'] = 1
        pressure_part['forward_pressure_count'] = pressure_part['forward_angle_flag']
        possession_parts.append(pressure_part.drop(columns=['forward_angle_flag']))

    if possession_parts:
        possession_df = pd.concat(possession_parts, ignore_index=True)
        possession_summary = (
            possession_df.groupby('possession', dropna=True)
            .agg(
                possession_min=('event_cont_min', 'min'),
                possession_max=('event_cont_min', 'max'),
                run_count=('run_count', 'sum'),
                run_distance_sum=('distance', 'sum'),
                shot_count=('shot_count', 'sum'),
                linked_pressure_count=('linked_pressure_count', 'sum'),
                forward_pressure_count=('forward_pressure_count', 'sum'),
            )
            .reset_index()
        )
        possession_summary['event_span'] = (
            possession_summary['possession_max'] - possession_summary['possession_min'] + 1
        )
    else:
        possession_summary = pd.DataFrame(
            columns=[
                'possession',
                'possession_min',
                'possession_max',
                'run_count',
                'run_distance_sum',
                'shot_count',
                'linked_pressure_count',
                'forward_pressure_count',
                'event_span',
            ]
        )

    if not possession_summary.empty:
        possession_export = possession_summary.copy()
        possession_export.insert(0, 'player_appearance_id', player_appearance_id)
        possession_export.insert(1, 'checkpoint', row.checkpoint)
        possession_export.insert(2, 'fixture_id', row.fixture_id)
        possession_export.insert(3, 'checkpoint_cont_min', checkpoint_time)
        possession_sequence_rows.extend(possession_export.to_dict(orient='records'))

    sequence_rows.append(
        {
            'player_appearance_id': player_appearance_id,
            'checkpoint': row.checkpoint,
            'fixture_id': row.fixture_id,
            'checkpoint_cont_min': checkpoint_time,
            'seq15m_pressure_total': int(minute_df['pressure_count'].sum()),
            'seq15m_pressure_turnover_total': int(minute_df['pressure_turnover_count'].sum()),
            'seq15m_run_total': int(minute_df['run_count'].sum()),
            'seq15m_run_distance_total': float(minute_df['run_distance_sum'].sum()),
            'seq15m_shot_total': int(minute_df['shot_count'].sum()),
            'seq15m_shot_under_pressure_total': int(minute_df['shot_under_pressure_count'].sum()),
            'seq15m_pressure_angle_availability_rate': safe_rate(
                int(minute_df['pressure_angle_observed_count'].sum()),
                int(minute_df['pressure_count'].sum()),
            ),
            'seq15m_forward_angle_share': safe_rate(
                int(pressure_last15['forward_angle_flag'].sum()),
                int(pressure_last15['pass_angle_observed'].sum()),
            ),
            'seq15m_lateral_angle_share': safe_rate(
                int(pressure_last15['lateral_angle_flag'].sum()),
                int(pressure_last15['pass_angle_observed'].sum()),
            ),
            'seq15m_backward_angle_share': safe_rate(
                int(pressure_last15['backward_angle_flag'].sum()),
                int(pressure_last15['pass_angle_observed'].sum()),
            ),
            'seq15m_angle_concentration': angle_concentration,
            'seq15m_angle_dispersion': 1.0 - angle_concentration if observed_angles.size else 0.0,
            'seq15m_directional_change_deg': float(directional_change_deg),
            'seq15m_pressured_minutes_with_runs_share': safe_rate(
                int(((minute_df['pressure_count'] > 0) & (minute_df['run_count'] > 0)).sum()),
                int((minute_df['pressure_count'] > 0).sum()),
            ),
            'seq15m_run_distance_after_pressure_rich_mean': float(rich_following_run_distance.mean())
            if len(rich_following_run_distance)
            else 0.0,
            'seq15m_shots_same_or_next_min_after_pressure': int(
                minute_df.loc[pressure_or_previous_mask, 'shot_count'].sum()
            ),
            'seq15m_under_pressure_shot_rate_after_forward_pressure': safe_rate(
                int(forward_up_shot_counts.sum()),
                int(forward_shot_counts.sum()),
            ),
            'seq15m_late_burst_count': int(activity_total.tail(5).ge(LATE_BURST_THRESHOLD).sum()),
            'seq15m_longest_active_streak': longest_true_streak(activity_total > 0),
            'seq15m_active_minute_share': float((activity_total > 0).mean()),
            'seq15m_recency_weighted_pressure_total': float(
                np.dot(minute_df['pressure_count'].to_numpy(dtype=float), weights)
            ),
            'seq15m_recency_weighted_run_total': float(
                np.dot(minute_df['run_count'].to_numpy(dtype=float), weights)
            ),
            'seq15m_recency_weighted_shot_total': float(
                np.dot(minute_df['shot_count'].to_numpy(dtype=float), weights)
            ),
            'seq15m_pressure_density_slope': linear_slope(
                minute_df['pressure_count'].to_numpy(dtype=float)
            ),
            'seq15m_run_intensity_slope': linear_slope(
                minute_df['run_distance_sum'].to_numpy(dtype=float)
            ),
            'seq15m_possession_transition_total': int(minute_df['possession_transition_count'].sum()),
            'seqpos_possession_count': int(len(possession_summary)),
            'seqpos_possessions_with_shots_share': float(
                (possession_summary['shot_count'] > 0).mean()
            )
            if not possession_summary.empty
            else 0.0,
            'seqpos_possessions_with_linked_pressure_share': float(
                (possession_summary['linked_pressure_count'] > 0).mean()
            )
            if not possession_summary.empty
            else 0.0,
            'seqpos_mean_runs_per_possession': float(possession_summary['run_count'].mean())
            if not possession_summary.empty
            else 0.0,
            'seqpos_mean_run_distance_per_possession': float(
                possession_summary['run_distance_sum'].mean()
            )
            if not possession_summary.empty
            else 0.0,
            'seqpos_max_run_distance_per_possession': float(
                possession_summary['run_distance_sum'].max()
            )
            if not possession_summary.empty
            else 0.0,
            'seqpos_mean_shots_per_possession': float(possession_summary['shot_count'].mean())
            if not possession_summary.empty
            else 0.0,
            'seqpos_pressure_shot_escalation_share': float(
                ((possession_summary['linked_pressure_count'] > 0) & (possession_summary['shot_count'] > 0)).mean()
            )
            if not possession_summary.empty
            else 0.0,
            'seqpos_forward_pressure_shot_share': float(
                ((possession_summary['forward_pressure_count'] > 0) & (possession_summary['shot_count'] > 0)).mean()
            )
            if not possession_summary.empty
            else 0.0,
            'seqpos_linked_pressure_total': int(possession_summary['linked_pressure_count'].sum())
            if not possession_summary.empty
            else 0,
            'seqpos_mean_possession_span': float(possession_summary['event_span'].mean())
            if not possession_summary.empty
            else 0.0,
            'seqpos_unmatched_pressure_count': int(len(unmatched_pressure_pre)),
            'seqpos_unmatched_pressure_share': safe_rate(int(len(unmatched_pressure_pre)), total_pressure_pre),
            'seqpos_unmatched_turnover_share': float(unmatched_pressure_pre['turnover_flag'].mean())
            if not unmatched_pressure_pre.empty
            else 0.0,
            'seqpos_unmatched_angle_availability_rate': float(
                unmatched_pressure_pre['pass_angle_observed'].mean()
            )
            if not unmatched_pressure_pre.empty
            else 0.0,
            'seqpos_strict_link_rate': safe_rate(int(linked_pressure_pre.shape[0]), total_pressure_pre),
            '_seq15m_max_event_cont_min': max(
                [
                    pressure_last15['event_cont_min'].max() if not pressure_last15.empty else 0,
                    runs_last15['event_cont_min'].max() if not runs_last15.empty else 0,
                    shots_last15['event_cont_min'].max() if not shots_last15.empty else 0,
                ]
            ),
            '_seqpos_max_event_cont_min': max(
                [
                    pressure_pre['event_cont_min'].max() if not pressure_pre.empty else 0,
                    runs_pre['event_cont_min'].max() if not runs_pre.empty else 0,
                    shots_pre['event_cont_min'].max() if not shots_pre.empty else 0,
                ]
            ),
        }
    )

sequence_feature_table = pd.DataFrame(sequence_rows).sort_values(
    ['fixture_id', 'player_appearance_id', 'checkpoint_cont_min']
).reset_index(drop=True)
minute_sequence_long = pd.DataFrame(minute_sequence_rows).sort_values(
    ['fixture_id', 'player_appearance_id', 'checkpoint_cont_min', 'minute_bin_cont_min']
).reset_index(drop=True)
possession_sequence_long = pd.DataFrame(possession_sequence_rows).sort_values(
    ['fixture_id', 'player_appearance_id', 'checkpoint_cont_min', 'possession_min', 'possession']
).reset_index(drop=True)

display(Markdown('### Sequence Feature Table Shape'))
display(
    pd.DataFrame(
        [
            {'output': 'sequence_feature_table', 'rows': len(sequence_feature_table), 'columns': sequence_feature_table.shape[1]},
            {'output': 'minute_sequence_long', 'rows': len(minute_sequence_long), 'columns': minute_sequence_long.shape[1]},
            {'output': 'possession_sequence_long', 'rows': len(possession_sequence_long), 'columns': possession_sequence_long.shape[1]},
        ]
    )
)
display(sequence_feature_table.head())

### Sequence Feature Table Shape

,output,rows,columns
0,sequence_feature_table,3486,48
1,minute_sequence_long,52290,25
2,possession_sequence_long,6448,13


,player_appearance_id,checkpoint,fixture_id,checkpoint_cont_min,seq15m_pressure_total,seq15m_pressure_turnover_total,seq15m_run_total,seq15m_run_distance_total,seq15m_shot_total,seq15m_shot_under_pressure_total,...,seqpos_forward_pressure_shot_share,seqpos_linked_pressure_total,seqpos_mean_possession_span,seqpos_unmatched_pressure_count,seqpos_unmatched_pressure_share,seqpos_unmatched_turnover_share,seqpos_unmatched_angle_availability_rate,seqpos_strict_link_rate,_seq15m_max_event_cont_min,_seqpos_max_event_cont_min
0,39421,H1_15,1159,15,0,0,0,0.000000,0,0,...,0.0,0,0.0,0,0.0,0.0,0.0,0.0,0,0
1,39421,H1_30,1159,30,0,0,0,0.000000,0,0,...,0.0,0,0.0,0,0.0,0.0,0.0,0.0,0,0
2,39421,H1_45,1159,45,0,0,2,19.051092,0,0,...,0.0,0,1.0,0,0.0,0.0,0.0,0.0,40,40
3,39421,H2_15,1159,60,0,0,0,0.000000,0,0,...,0.0,0,1.0,0,0.0,0.0,0.0,0.0,0,40
4,39421,H2_30,1159,75,0,0,0,0.000000,0,0,...,0.0,0,1.0,0,0.0,0.0,0.0,0.0,0,40


In [5]:
sequence_feature_export = sequence_feature_table.drop(
    columns=['_seq15m_max_event_cont_min', '_seqpos_max_event_cont_min']
)
modeling_table = base.merge(
    sequence_feature_export,
    on=['player_appearance_id', 'checkpoint', 'fixture_id', 'checkpoint_cont_min'],
    how='left',
    validate='one_to_one',
)

validation_checks = pd.DataFrame(
    [
        {
            'check_name': 'seq15m_future_leak_rows',
            'value': int(
                (
                    sequence_feature_table['_seq15m_max_event_cont_min']
                    > sequence_feature_table['checkpoint_cont_min']
                ).sum()
            ),
            'expected': 0,
        },
        {
            'check_name': 'seqpos_future_leak_rows',
            'value': int(
                (
                    sequence_feature_table['_seqpos_max_event_cont_min']
                    > sequence_feature_table['checkpoint_cont_min']
                ).sum()
            ),
            'expected': 0,
        },
        {
            'check_name': 'join_row_delta',
            'value': int(len(modeling_table) - len(base)),
            'expected': 0,
        },
        {
            'check_name': 'missing_sequence_rows_after_join',
            'value': int(modeling_table['seq15m_pressure_total'].isna().sum()),
            'expected': 0,
        },
        {
            'check_name': 'ambiguous_pressure_rows',
            'value': int((pressure['possession_nunique'].fillna(0) > 1).sum()),
            'expected': 'diagnostic',
        },
        {
            'check_name': 'ambiguous_pressure_rows_linked',
            'value': int(
                ((pressure['possession_nunique'].fillna(0) > 1) & pressure['linked_pressure_flag'].eq(1)).sum()
            ),
            'expected': 0,
        },
    ]
)
validation_checks['status'] = np.where(
    validation_checks['expected'].astype(str) == 'diagnostic',
    'diagnostic',
    np.where(validation_checks['value'] == validation_checks['expected'], 'pass', 'fail'),
)

sequence_feature_columns = [
    column
    for column in sequence_feature_export.columns
    if column.startswith('seq15m_') or column.startswith('seqpos_')
]
feature_support = pd.DataFrame(
    [
        {
            'feature': column,
            'family': 'seq15m' if column.startswith('seq15m_') else 'seqpos',
            'nonzero_rate': float((sequence_feature_export[column] != 0).mean()),
            'mean': float(sequence_feature_export[column].mean()),
            'std': float(sequence_feature_export[column].std(ddof=0)),
        }
        for column in sequence_feature_columns
    ]
).sort_values(['family', 'nonzero_rate', 'feature'], ascending=[True, False, True]).reset_index(drop=True)

display(Markdown('### Validation Checks'))
display(validation_checks)
display(Markdown('### Feature Support Snapshot'))
display(feature_support.head(15))

### Validation Checks

,check_name,value,expected,status
0,seq15m_future_leak_rows,0,0,pass
1,seqpos_future_leak_rows,0,0,pass
2,join_row_delta,0,0,pass
3,missing_sequence_rows_after_join,0,0,pass
4,ambiguous_pressure_rows,919,diagnostic,diagnostic
5,ambiguous_pressure_rows_linked,0,0,pass


### Feature Support Snapshot

,feature,family,nonzero_rate,mean,std
0,seq15m_active_minute_share,seq15m,0.946644,0.422872,0.198270
1,seq15m_longest_active_streak,seq15m,0.946644,2.813827,1.668631
2,seq15m_recency_weighted_run_total,seq15m,0.930293,4.117499,2.583137
3,seq15m_run_distance_total,seq15m,0.930293,367.125082,978.629056
4,seq15m_run_total,seq15m,0.930293,7.938325,4.798933
5,seq15m_run_intensity_slope,seq15m,0.930006,-0.650161,6.366789
6,seq15m_pressure_total,seq15m,0.820138,2.915089,2.614277
7,seq15m_recency_weighted_pressure_total,seq15m,0.820138,1.507669,1.458507
8,seq15m_pressure_density_slope,seq15m,0.812966,-0.002520,0.030454
9,seq15m_angle_concentration,seq15m,0.660929,0.577392,0.436994


In [6]:
development_df = modeling_table[modeling_table['partition_role'] == 'development'].copy().reset_index(drop=True)

feature_sets = {
    'base_only': [],
    'base_plus_minute_hybrid': [column for column in modeling_table.columns if column.startswith('seq15m_')],
    'base_plus_possession_sequence': [column for column in modeling_table.columns if column.startswith('seqpos_')],
    'base_plus_both': [
        column
        for column in modeling_table.columns
        if column.startswith('seq15m_') or column.startswith('seqpos_')
    ],
}

metrics_rows = []
fold_metric_rows = []
oof_registry = {}

for model_name in ['logreg', 'rf']:
    for feature_set_name, extra_columns in feature_sets.items():
        feature_columns = BASE_FEATURE_COLUMNS + extra_columns
        fold_metrics_df, oof_df = evaluate_feature_set(
            development_df=development_df,
            feature_columns=feature_columns,
            model_name=model_name,
            return_oof=(model_name == 'logreg' and feature_set_name == 'base_plus_both'),
        )
        summary_row = {
            'model_name': model_name,
            'feature_set': feature_set_name,
            'n_features': len(feature_columns),
        }
        for metric_name in ['balanced_accuracy', 'roc_auc', 'pr_auc', 'brier_score']:
            summary_row[metric_name] = float(fold_metrics_df[metric_name].mean())
        metrics_rows.append(summary_row)

        fold_metrics_export = fold_metrics_df.copy()
        fold_metrics_export.insert(0, 'model_name', model_name)
        fold_metrics_export.insert(1, 'feature_set', feature_set_name)
        fold_metric_rows.append(fold_metrics_export)

        if oof_df is not None:
            oof_registry[(model_name, feature_set_name)] = oof_df.copy()

metrics_table = pd.DataFrame(metrics_rows).sort_values(
    ['balanced_accuracy', 'roc_auc', 'pr_auc'],
    ascending=[False, False, False],
).reset_index(drop=True)
fold_metrics_table = pd.concat(fold_metric_rows, ignore_index=True)

display(Markdown('### Ablation Metrics'))
display(metrics_table)

### Ablation Metrics

,model_name,feature_set,n_features,balanced_accuracy,roc_auc,pr_auc,brier_score
0,logreg,base_only,25,0.583135,0.614308,0.094706,0.221995
1,logreg,base_plus_minute_hybrid,51,0.562430,0.599728,0.096606,0.219781
2,logreg,base_plus_both,67,0.560841,0.566554,0.085662,0.215815
3,logreg,base_plus_possession_sequence,41,0.560769,0.581339,0.082633,0.215798
4,rf,base_only,25,0.503287,0.648726,0.118565,0.072401
5,rf,base_plus_possession_sequence,41,0.503032,0.638927,0.107929,0.067540
6,rf,base_plus_both,67,0.501546,0.637287,0.112061,0.064720
7,rf,base_plus_minute_hybrid,51,0.499240,0.647417,0.114512,0.065599


In [7]:
best_logreg_feature_columns = BASE_FEATURE_COLUMNS + feature_sets['base_plus_both']
interpretation_pipeline = Pipeline(
    steps=[
        ('prep', build_preprocessor(best_logreg_feature_columns)),
        (
            'model',
            LogisticRegression(
                max_iter=4000,
                class_weight='balanced',
                solver='liblinear',
                C=0.25,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)
interpretation_pipeline.fit(
    development_df[best_logreg_feature_columns],
    development_df['scored_after'].astype(int),
)
coefficient_frame = pd.DataFrame(
    {
        'feature': interpretation_pipeline.named_steps['prep'].get_feature_names_out(),
        'coefficient': interpretation_pipeline.named_steps['model'].coef_[0],
    }
)
coefficient_frame['feature'] = coefficient_frame['feature'].str.replace('num__', '', regex=False).str.replace('cat__', '', regex=False)
coefficient_frame['abs_coefficient'] = coefficient_frame['coefficient'].abs()
interpretation_table = coefficient_frame[
    coefficient_frame['feature'].str.startswith('seq15m_')
    | coefficient_frame['feature'].str.startswith('seqpos_')
].sort_values('abs_coefficient', ascending=False).head(20).reset_index(drop=True)

subgroup_predictions = oof_registry[('logreg', 'base_plus_both')].copy()
subgroup_stability_table = pd.concat(
    [
        subgroup_metric_table(subgroup_predictions, 'checkpoint'),
        subgroup_metric_table(subgroup_predictions, 'position'),
    ],
    ignore_index=True,
)

display(Markdown('### Strongest Geometry And Sequence Effects'))
display(interpretation_table)
display(Markdown('### Subgroup Stability For `logreg + base_plus_both`'))
display(subgroup_stability_table)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


### Strongest Geometry And Sequence Effects

,feature,coefficient,abs_coefficient
0,seqpos_max_run_distance_per_possession,0.970016,0.970016
1,seqpos_mean_run_distance_per_possession,0.875379,0.875379
2,seq15m_run_distance_after_pressure_rich_mean,-0.828075,0.828075
3,seqpos_possessions_with_shots_share,0.484499,0.484499
4,seqpos_pressure_shot_escalation_share,-0.350512,0.350512
5,seqpos_linked_pressure_total,0.326453,0.326453
6,seq15m_pressure_turnover_total,0.317356,0.317356
7,seq15m_pressure_total,-0.249934,0.249934
8,seq15m_recency_weighted_pressure_total,-0.248841,0.248841
9,seqpos_mean_runs_per_possession,0.243598,0.243598


### Subgroup Stability For `logreg + base_plus_both`

,group_type,group_value,rows,positive_rate,balanced_accuracy,pr_auc,brier_score,roc_auc
0,checkpoint,H1_15,549,0.087432,0.573416,0.107738,0.298728,0.584165
1,checkpoint,H1_30,549,0.067395,0.586677,0.106924,0.242410,0.566987
2,checkpoint,H1_45,549,0.058288,0.480235,0.054435,0.234741,0.478965
3,checkpoint,H2_15,549,0.041894,0.486196,0.053175,0.158201,0.481485
4,checkpoint,H2_30,549,0.040073,0.584095,0.063361,0.155615,0.532086
5,checkpoint,H2_45,43,0.046512,0.378049,0.051587,0.171780,0.341463
6,checkpoint,ET1_15,42,0.023810,0.414634,0.029412,0.131171,0.195122
7,position,M,1050,0.069524,0.472035,0.060026,0.263484,0.425976
8,position,D,1009,0.028741,0.447449,0.021396,0.120627,0.349296
9,position,A,513,0.122807,0.503968,0.112501,0.407359,0.464056


In [8]:
sequence_feature_export.to_csv(ARTIFACT_DIR / 'sequence_feature_table.csv', index=False)
minute_sequence_long.to_csv(ARTIFACT_DIR / 'minute_sequence_long.csv', index=False)
possession_sequence_long.to_csv(ARTIFACT_DIR / 'possession_sequence_long.csv', index=False)
validation_checks.to_csv(ARTIFACT_DIR / 'validation_checks.csv', index=False)
feature_support.to_csv(ARTIFACT_DIR / 'feature_support.csv', index=False)
metrics_table.to_csv(ARTIFACT_DIR / 'metrics_table.csv', index=False)
fold_metrics_table.to_csv(ARTIFACT_DIR / 'fold_metrics_table.csv', index=False)
interpretation_table.to_csv(ARTIFACT_DIR / 'interpretation_table.csv', index=False)
subgroup_stability_table.to_csv(ARTIFACT_DIR / 'subgroup_stability_table.csv', index=False)

display(Markdown('### Exported Artifacts'))
display(
    pd.DataFrame(
        {
            'artifact': [
                'sequence_feature_table.csv',
                'minute_sequence_long.csv',
                'possession_sequence_long.csv',
                'validation_checks.csv',
                'feature_support.csv',
                'metrics_table.csv',
                'fold_metrics_table.csv',
                'interpretation_table.csv',
                'subgroup_stability_table.csv',
            ],
            'path': [str(ARTIFACT_DIR / name) for name in [
                'sequence_feature_table.csv',
                'minute_sequence_long.csv',
                'possession_sequence_long.csv',
                'validation_checks.csv',
                'feature_support.csv',
                'metrics_table.csv',
                'fold_metrics_table.csv',
                'interpretation_table.csv',
                'subgroup_stability_table.csv',
            ]],
        }
    )
)

### Exported Artifacts

,artifact,path
0,sequence_feature_table.csv,/Users/jan/Documents/competitions/hackatons/WE...
1,minute_sequence_long.csv,/Users/jan/Documents/competitions/hackatons/WE...
2,possession_sequence_long.csv,/Users/jan/Documents/competitions/hackatons/WE...
3,validation_checks.csv,/Users/jan/Documents/competitions/hackatons/WE...
4,feature_support.csv,/Users/jan/Documents/competitions/hackatons/WE...
5,metrics_table.csv,/Users/jan/Documents/competitions/hackatons/WE...
6,fold_metrics_table.csv,/Users/jan/Documents/competitions/hackatons/WE...
7,interpretation_table.csv,/Users/jan/Documents/competitions/hackatons/WE...
8,subgroup_stability_table.csv,/Users/jan/Documents/competitions/hackatons/WE...
